# Data Extraction
**Project:** F1 Constructor Sponsorship & Investment ROI Analytics  

**Objective:** Load all raw CSV files, profile their structure, filter to the Modern Era (2014–Present), and prepare a joined `master dataset` for cleaning.  


## 1. Setup


In [40]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.float_format', '{:.2f}'.format)



## 2. Load Raw Data
We load all 14 CSV files from `data/raw/`. These files are the **original, unedited** dataset as required by the Capstone Handbook.  
The `\N` values in the dataset represent missing entries and are parsed as NaN.


In [41]:
# Configure the data path
RAW_DATA_DIR = '../data/raw/'

# Load all CSV files
circuits = pd.read_csv(os.path.join(RAW_DATA_DIR, 'circuits.csv'), na_values=['\\N'])
constructors = pd.read_csv(os.path.join(RAW_DATA_DIR, 'constructors.csv'), na_values=['\\N'])
constructor_results = pd.read_csv(os.path.join(RAW_DATA_DIR, 'constructor_results.csv'), na_values=['\\N'])
constructor_standings = pd.read_csv(os.path.join(RAW_DATA_DIR, 'constructor_standings.csv'), na_values=['\\N'])
drivers = pd.read_csv(os.path.join(RAW_DATA_DIR, 'drivers.csv'), na_values=['\\N'])
driver_standings = pd.read_csv(os.path.join(RAW_DATA_DIR, 'driver_standings.csv'), na_values=['\\N'])
lap_times = pd.read_csv(os.path.join(RAW_DATA_DIR, 'lap_times.csv'), na_values=['\\N'])
pit_stops = pd.read_csv(os.path.join(RAW_DATA_DIR, 'pit_stops.csv'), na_values=['\\N'])
qualifying = pd.read_csv(os.path.join(RAW_DATA_DIR, 'qualifying.csv'), na_values=['\\N'])
races = pd.read_csv(os.path.join(RAW_DATA_DIR, 'races.csv'), na_values=['\\N'])
results = pd.read_csv(os.path.join(RAW_DATA_DIR, 'results.csv'), na_values=['\\N'])
seasons = pd.read_csv(os.path.join(RAW_DATA_DIR, 'seasons.csv'), na_values=['\\N'])
sprint_results = pd.read_csv(os.path.join(RAW_DATA_DIR, 'sprint_results.csv'), na_values=['\\N'])
status = pd.read_csv(os.path.join(RAW_DATA_DIR, 'status.csv'), na_values=['\\N'])

print('All 14 raw CSV files loaded successfully.')


All 14 raw CSV files loaded successfully.


## 3. Initial Shape & Size Overview


In [42]:
# Build a summary table of all datasets
datasets = {
    'circuits': circuits, 'constructors': constructors,
    'constructor_results': constructor_results, 'constructor_standings': constructor_standings,
    'drivers': drivers, 'driver_standings': driver_standings,
    'lap_times': lap_times, 'pit_stops': pit_stops,
    'qualifying': qualifying, 'races': races,
    'results': results, 'seasons': seasons,
    'sprint_results': sprint_results, 'status': status
}

summary = []
for name, df in datasets.items():
    summary.append({
        'Table': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Null Cells': df.isna().sum().sum(),
        'Null %': round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 1),
        'Memory (KB)': round(df.memory_usage(deep=True).sum() / 1024, 1)
    })

summary_df = pd.DataFrame(summary)
print(f"Total rows across all tables: {summary_df['Rows'].sum():,}")
print(f"Total columns across all tables: {summary_df['Columns'].sum()}")
print()
summary_df


Total rows across all tables: 701,433
Total columns across all tables: 120



,Table,Rows,Columns,Null Cells,Null %,Memory (KB)
0,circuits,77,9,0,0.00,31.20
1,constructors,212,5,0,0.00,62.80
2,constructor_results,12625,5,12608,20.00,789.60
3,constructor_standings,13391,7,0,0.00,1390.10
4,drivers,861,9,1559,20.10,400.60
5,driver_standings,34863,7,0,0.00,3633.20
6,lap_times,589081,6,0,0.00,60404.60
7,pit_stops,11371,7,0,0.00,1867.10
8,qualifying,10494,9,11668,12.40,2114.40
9,races,1125,18,11349,56.00,733.60


## 4. Column Profiling
We inspect each critical table for data types, null counts, and sample values.  
This information directly feeds into our **Data Dictionary**.


In [43]:
def profile_table(df, name):
    """Print a detailed column-by-column profile of the dataframe."""
    print(f'=== {name.upper()} ({df.shape[0]:,} rows x {df.shape[1]} cols) ===')
    print()
    
    profile_data = []
    for col in df.columns:
        non_null = df[col].notna().sum()
        null_count = df[col].isna().sum()
        null_pct = round(null_count / len(df) * 100, 1)
        unique = df[col].nunique()
        samples = df[col].dropna().unique()[:3]
        sample_str = ', '.join([str(s) for s in samples])[:45]
        
        profile_data.append({
            'Column': col, 'Type': str(df[col].dtype),
            'Non-Null': non_null, 'Nulls': null_count,
            'Null%': null_pct, 'Unique': unique, 'Samples': sample_str
        })
    
    return pd.DataFrame(profile_data)

# Profile the 6 most critical tables
for name in ['races', 'results', 'constructors', 'constructor_standings', 'pit_stops', 'status']:
    display(profile_table(datasets[name], name))
    print()


=== RACES (1,125 rows x 18 cols) ===



,Column,Type,Non-Null,Nulls,Null%,Unique,Samples
0,raceId,int64,1125,0,0.00,1125,"1, 2, 3"
1,year,int64,1125,0,0.00,75,"2009, 2008, 2007"
2,round,int64,1125,0,0.00,24,"1, 2, 3"
3,circuitId,int64,1125,0,0.00,77,"1, 2, 17"
4,name,object,1125,0,0.00,54,"Australian Grand Prix, Malaysian Gra..."
5,date,object,1125,0,0.00,1125,"2009-03-29, 2009-04-05, 2009-04-19"
6,time,object,394,731,65.00,34,"06:00:00, 09:00:00, 07:00:00"
7,url,object,1125,0,0.00,1125,http://en.wikipedia.org/wiki/2009_Au...
8,fp1_date,object,90,1035,92.00,90,"2021-04-16, 2022-03-18, 2021-03-26"
9,fp1_time,object,68,1057,94.00,20,"12:00:00, 14:00:00, 03:00:00"



=== RESULTS (26,759 rows x 18 cols) ===



,Column,Type,Non-Null,Nulls,Null%,Unique,Samples
0,resultId,int64,26759,0,0.00,26759,"1, 2, 3"
1,raceId,int64,26759,0,0.00,1125,"18, 19, 20"
2,driverId,int64,26759,0,0.00,861,"1, 2, 3"
3,constructorId,int64,26759,0,0.00,211,"1, 2, 3"
4,number,float64,26753,6,0.00,129,"22.0, 3.0, 7.0"
5,grid,int64,26759,0,0.00,35,"1, 5, 7"
6,position,float64,15806,10953,40.90,33,"1.0, 2.0, 3.0"
7,positionText,object,26759,0,0.00,39,"1, 2, 3"
8,positionOrder,int64,26759,0,0.00,39,"1, 2, 3"
9,points,float64,26759,0,0.00,39,"10.0, 8.0, 6.0"



=== CONSTRUCTORS (212 rows x 5 cols) ===



...

,Column,Type,Non-Null,Nulls,Null%,Unique,Samples
0,constructorStandingsId,int64,13391,0,0.00,13391,"1, 2, 3"
1,raceId,int64,13391,0,0.00,1061,"18, 19, 20"
2,constructorId,int64,13391,0,0.00,160,"1, 2, 3"
3,points,float64,13391,0,0.00,579,"14.0, 8.0, 9.0"
4,position,int64,13391,0,0.00,22,"1, 3, 2"
5,positionText,object,13391,0,0.00,23,"1, 3, 2"
6,wins,int64,13391,0,0.00,22,"1, 0, 2"



=== PIT_STOPS (11,371 rows x 7 cols) ===



,Column,Type,Non-Null,Nulls,Null%,Unique,Samples
0,raceId,int64,11371,0,0.00,285,"841, 842, 843"
1,driverId,int64,11371,0,0.00,76,"153, 30, 17"
2,stop,int64,11371,0,0.00,14,"1, 2, 3"
3,lap,int64,11371,0,0.00,74,"1, 11, 12"
4,time,object,11371,0,0.00,8227,"17:05:23, 17:05:52, 17:20:48"
5,duration,object,11371,0,0.00,7604,"26.898, 25.021, 23.426"
6,milliseconds,int64,11371,0,0.00,7604,"26898, 25021, 23426"



=== STATUS (139 rows x 2 cols) ===



,Column,Type,Non-Null,Nulls,Null%,Unique,Samples
0,statusId,int64,139,0,0.00,139,"1, 2, 3"
1,status,object,139,0,0.00,139,"Finished, Disqualified, Accident"


## 5. Filter (2014-Present)
The 2014 season marked the biggest regulatory change in F1 history - the switch from 2.4L V8 engines to 1.6L V6 Turbo Hybrids. Data before and after 2014 is fundamentally different and not comparable.

We filter `races` to `year >= 2014` and use the resulting `raceId` list to filter all other tables.


In [44]:
# Filter races to 2014 onwards
modern_races = races[races['year'] >= 2014].copy()
modern_race_ids = set(modern_races['raceId'].unique())

print(f'Races in full dataset:    {len(races):,}')
print(f'Races from 2014 onwards:  {len(modern_races):,}')
print(f'Seasons covered:          {modern_races["year"].min()} to {modern_races["year"].max()}')
print(f'Number of seasons:        {modern_races["year"].nunique()}')
print()
print('Races per season:')
print(modern_races.groupby('year')['raceId'].count().to_string())


Races in full dataset:    1,125
Races from 2014 onwards:  228
Seasons covered:          2014 to 2024
Number of seasons:        11

Races per season:
year
2014    19
2015    19
2016    21
2017    20
2018    21
2019    21
2020    17
2021    22
2022    22
2023    22
2024    24


### 5.1 Apply the Era Filter to All Tables


In [45]:
# Filter all tables using the modern raceId set
modern_results = results[results['raceId'].isin(modern_race_ids)].copy()
modern_constructor_results = constructor_results[constructor_results['raceId'].isin(modern_race_ids)].copy()
modern_constructor_standings = constructor_standings[constructor_standings['raceId'].isin(modern_race_ids)].copy()
modern_pit_stops = pit_stops[pit_stops['raceId'].isin(modern_race_ids)].copy()
modern_qualifying = qualifying[qualifying['raceId'].isin(modern_race_ids)].copy()
modern_lap_times = lap_times[lap_times['raceId'].isin(modern_race_ids)].copy()
modern_sprint_results = sprint_results[sprint_results['raceId'].isin(modern_race_ids)].copy()

# Lookup tables stay as-is (no raceId to filter)
# constructors, drivers, circuits, status — remain unfiltered

# Post-filter shape comparison
filter_summary = pd.DataFrame([
    {'Table': 'races', 'Before': len(races), 'After': len(modern_races)},
    {'Table': 'results', 'Before': len(results), 'After': len(modern_results)},
    {'Table': 'constructor_results', 'Before': len(constructor_results), 'After': len(modern_constructor_results)},
    {'Table': 'constructor_standings', 'Before': len(constructor_standings), 'After': len(modern_constructor_standings)},
    {'Table': 'pit_stops', 'Before': len(pit_stops), 'After': len(modern_pit_stops)},
    {'Table': 'qualifying', 'Before': len(qualifying), 'After': len(modern_qualifying)},
    {'Table': 'lap_times', 'Before': len(lap_times), 'After': len(modern_lap_times)},
    {'Table': 'sprint_results', 'Before': len(sprint_results), 'After': len(modern_sprint_results)},
])
filter_summary['Retained %'] = round(filter_summary['After'] / filter_summary['Before'] * 100, 1)
filter_summary


,Table,Before,After,Retained %
0,races,1125,228,20.30
1,results,26759,4626,17.30
2,constructor_results,12625,2318,18.40
3,constructor_standings,13391,2319,17.30
4,pit_stops,11371,8360,73.50
5,qualifying,10494,4610,43.90
6,lap_times,589081,248144,42.10
7,sprint_results,360,360,100.00


## 6. Post-Filter Null Analysis
Now that we have only the modern era data from 2014 + , let us re-check the null situation. 


In [46]:
# Check nulls in the modern results table (our most important table)
print('=== MODERN RESULTS — NULL ANALYSIS ===')
print()
null_info = pd.DataFrame({
    'Column': modern_results.columns,
    'Non-Null': modern_results.notna().sum().values,
    'Nulls': modern_results.isna().sum().values,
    'Null%': round(modern_results.isna().sum() / len(modern_results) * 100, 1).values,
    'Dtype': modern_results.dtypes.astype(str).values
})
null_info[null_info['Nulls'] > 0]


=== MODERN RESULTS — NULL ANALYSIS ===



,Column,Non-Null,Nulls,Null%,Dtype
6,position,3905,721,15.60,float64
11,time,2407,2219,48.00,object
12,milliseconds,2407,2219,48.00,float64
13,fastestLap,4409,217,4.70,float64
15,fastestLapTime,4409,217,4.70,object
16,fastestLapSpeed,4409,217,4.70,float64


## 7. Identify Active Constructors (2014+)
Not all 212 constructors in the dataset raced in the modern era. We extract only the ones that appear in the 2014+ results.


In [47]:
# Get unique constructorIds from modern era
active_constructor_ids = modern_results['constructorId'].unique()
active_constructors = constructors[constructors['constructorId'].isin(active_constructor_ids)].copy()

print(f'Total constructors in dataset: {len(constructors)}')
print(f'Active constructors (2014+):   {len(active_constructors)}')
print()
print('Teams racing in the Modern Era:')
for _, row in active_constructors.sort_values('name').iterrows():
    print(f'  {row["constructorId"]:>3}  {row["name"]}')


Total constructors in dataset: 212
Active constructors (2014+):   20

Teams racing in the Modern Era:
   51  Alfa Romeo
  213  AlphaTauri
  214  Alpine F1 Team
  117  Aston Martin
  207  Caterham
    6  Ferrari
   10  Force India
  210  Haas F1 Team
  208  Lotus F1
  209  Manor Marussia
  206  Marussia
    1  McLaren
  131  Mercedes
  215  RB F1 Team
  211  Racing Point
    9  Red Bull
    4  Renault
   15  Sauber
    5  Toro Rosso
    3  Williams


In [48]:
# Similarly for drivers
active_driver_ids = modern_results['driverId'].unique()
active_drivers = drivers[drivers['driverId'].isin(active_driver_ids)].copy()

print(f'Total drivers in dataset: {len(drivers)}')
print(f'Active drivers (2014+):   {len(active_drivers)}')


Total drivers in dataset: 861
Active drivers (2014+):   59


## 8. Build Master Joined Dataset
We join the key tables together into a single "master" dataframe that the cleaning notebook will process.

**Join Strategy:**
1. Start with `modern_results` (core)
2. Join `modern_races` to get `year`, `round`, `circuitId`, `race_name`, `date`
3. Join `constructors` to get `constructor_name`, `nationality`
4. Join `drivers` to get `forename`, `surname`, `driver_nationality`
5. Join `status` to get the text status (Finished, Accident, etc.)
6. Join `circuits` to get `circuit_name`, `country`, `lat`, `lng`


In [49]:
# Step 1: Start with modern_results
master = modern_results.copy()

# Step 2: Join races (to get year, round, date, circuit)
races_cols = modern_races[['raceId', 'year', 'round', 'circuitId', 'name', 'date']].copy()
races_cols = races_cols.rename(columns={'name': 'race_name', 'date': 'race_date'})
master = master.merge(races_cols, on='raceId', how='left')

# Step 3: Join constructors (to get team names)
const_cols = constructors[['constructorId', 'name', 'nationality']].copy()
const_cols = const_cols.rename(columns={'name': 'constructor_name', 'nationality': 'constructor_nationality'})
master = master.merge(const_cols, on='constructorId', how='left')

# Step 4: Join drivers (to get driver names)
driver_cols = drivers[['driverId', 'forename', 'surname', 'nationality', 'dob']].copy()
driver_cols = driver_cols.rename(columns={'nationality': 'driver_nationality'})
master = master.merge(driver_cols, on='driverId', how='left')

# Step 5: Join status (to get finish status text)
master = master.merge(status, on='statusId', how='left')

# Step 6: Join circuits (to get location info)
circuit_cols = circuits[['circuitId', 'name', 'location', 'country', 'lat', 'lng']].copy()
circuit_cols = circuit_cols.rename(columns={'name': 'circuit_name', 'location': 'circuit_location', 'country': 'circuit_country'})
master = master.merge(circuit_cols, on='circuitId', how='left')

print(f'Master dataset shape: {master.shape}')
print(f'Columns: {list(master.columns)}')
print()
master.head()


Master dataset shape: (4626, 35)
Columns: ['resultId', 'raceId', 'driverId', 'constructorId', 'number', 'grid', 'position', 'positionText', 'positionOrder', 'points', 'laps', 'time', 'milliseconds', 'fastestLap', 'rank', 'fastestLapTime', 'fastestLapSpeed', 'statusId', 'year', 'round', 'circuitId', 'race_name', 'race_date', 'constructor_name', 'constructor_nationality', 'forename', 'surname', 'driver_nationality', 'dob', 'status', 'circuit_name', 'circuit_location', 'circuit_country', 'lat', 'lng']



,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId,year,round,circuitId,race_name,race_date,constructor_name,constructor_nationality,forename,surname,driver_nationality,dob,status,circuit_name,circuit_location,circuit_country,lat,lng
0,22130,900,3,131,6.00,3,1.00,1,1,25.00,57,1:32:58.710,5578710.00,19.00,1.00,1:32.478,206.44,1,2014,1,1,Australian Grand Prix,2014-03-16,Mercedes,German,Nico,Rosberg,German,1985-06-27,Finished,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.85,144.97
1,22131,900,825,1,20.00,4,2.00,2,2,18.00,57,+26.777,5605487.00,49.00,6.00,1:33.066,205.13,1,2014,1,1,Australian Grand Prix,2014-03-16,McLaren,British,Kevin,Magnussen,Danish,1992-10-05,Finished,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.85,144.97
2,22132,900,18,1,22.00,10,3.00,3,3,15.00,57,+30.027,5608737.00,39.00,5.00,1:32.917,205.46,1,2014,1,1,Australian Grand Prix,2014-03-16,McLaren,British,Jenson,Button,British,1980-01-19,Finished,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.85,144.97
3,22133,900,4,6,14.00,5,4.00,4,4,12.00,57,+35.284,5613994.00,57.00,7.00,1:33.186,204.87,1,2014,1,1,Australian Grand Prix,2014-03-16,Ferrari,Italian,Fernando,Alonso,Spanish,1981-07-29,Finished,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.85,144.97
4,22134,900,822,3,77.00,15,5.00,5,5,10.00,57,+47.639,5626349.00,56.00,3.00,1:32.616,206.13,1,2014,1,1,Australian Grand Prix,2014-03-16,Williams,British,Valtteri,Bottas,Finnish,1989-08-28,Finished,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.85,144.97


### 8.1 Verify the Master Table


In [50]:
# Quick sanity checks
print('=== MASTER TABLE VERIFICATION ===')
print(f'Total rows:        {len(master):,}')
print(f'Total columns:     {master.shape[1]}')
print(f'Year range:        {master["year"].min()} to {master["year"].max()}')
print(f'Unique races:      {master["raceId"].nunique()}')
print(f'Unique drivers:    {master["driverId"].nunique()}')
print(f'Unique teams:      {master["constructorId"].nunique()}')
print(f'Unique circuits:   {master["circuitId"].nunique()}')
print()

# Check for any failed joins (nulls introduced by merge)
join_nulls = master[['race_name', 'constructor_name', 'surname', 'status', 'circuit_name']].isna().sum()
print('Null check after joins (should be 0):')
print(join_nulls)


=== MASTER TABLE VERIFICATION ===
Total rows:        4,626
Total columns:     35
Year range:        2014 to 2024
Unique races:      228
Unique drivers:    59
Unique teams:      20
Unique circuits:   32

Null check after joins (should be 0):
race_name           0
constructor_name    0
surname             0
status              0
circuit_name        0
dtype: int64


## 9. Save Extracted Data
We save three outputs:
1. **`master_extracted.csv`** — The full joined table (results + races + teams + drivers + status + circuits)
2. **`modern_pit_stops.csv`** — Filtered pit stops (kept separate due to different granularity)
3. **`modern_constructor_standings.csv`** — Constructor championship standings (needed for YoY Growth KPI)

These go to a  `data/extracted` folder. The cleaning notebook will pick them up.


In [51]:
# Create output directory
EXTRACT_DIR = '../data/extracted/'
os.makedirs(EXTRACT_DIR, exist_ok=True)

# Save master table
master.to_csv(os.path.join(EXTRACT_DIR, 'master_extracted.csv'), index=False)
print(f'Saved master_extracted.csv ({len(master):,} rows x {master.shape[1]} cols)')

# Save pit stops separately
modern_pit_stops.to_csv(os.path.join(EXTRACT_DIR, 'modern_pit_stops.csv'), index=False)
print(f'Saved modern_pit_stops.csv ({len(modern_pit_stops):,} rows x {modern_pit_stops.shape[1]} cols)')

# Save constructor standings separately (needed for YoY Growth KPI)
modern_constructor_standings.to_csv(os.path.join(EXTRACT_DIR, 'modern_constructor_standings.csv'), index=False)
print(f'Saved modern_constructor_standings.csv ({len(modern_constructor_standings):,} rows x {modern_constructor_standings.shape[1]} cols)')

print()
print('Extraction phase complete. Proceed to 02_cleaning.ipynb.')


Saved master_extracted.csv (4,626 rows x 35 cols)
Saved modern_pit_stops.csv (8,360 rows x 7 cols)
Saved modern_constructor_standings.csv (2,319 rows x 7 cols)

Extraction phase complete. Proceed to 02_cleaning.ipynb.


In [52]:
# Extraction and Filtering is done .